In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("..")

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
import tqdm

from plots import hist_events_by_labels
from events_data import EventsData
from fvt_classifier import FvTClassifier
# import LogNorm
from matplotlib.colors import LogNorm
from training_info import TrainingInfo
from attention_classifier import AttentionClassifier
from plots import plot_rewighted_samples_by_model, plot_samples_raw
from dataset import MotherSamples
from events_data import events_from_scdinfo



features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

# use tex
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.titlesize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["lines.markersize"] = 3

import pandas as pd


df_3b = pd.read_hdf("../events/MG3/dataframes/threeTag_picoAOD.h5")
df_bg4b = pd.read_hdf("../events/MG3/dataframes/fourTag_10x_picoAOD.h5")
df_signal = pd.read_hdf("../events/MG3/dataframes/HH4b_picoAOD.h5")
df_3b["signal"] = False
df_bg4b["signal"] = False
df_signal["signal"] = True
raw_df_list = [df_3b, df_bg4b, df_signal]

In [4]:
metadata = TrainingInfo.load_metadata()

In [5]:
corrupted_hash = "241228_231134_595452_3Dm6Ge"
metadata[corrupted_hash]

impacted_experiment_names = [
    "CR_fvt_training_ensemble_max_smeared", 
    "CR_fvt_training_ensemble_max_fvt"
]

for experiment_name in impacted_experiment_names:
    hashes = TrainingInfo.find({"experiment_name": experiment_name})
    for hash in hashes:
        tinfo = TrainingInfo.load(hash)
        if corrupted_hash in tinfo.hparams["signal_region"]["SR_stats_hashes"]:
            print(f"Hash {hash} is impacted by {corrupted_hash}")
            
        

Hash 250102_171037_542935_JJwOaT is impacted by 241228_231134_595452_3Dm6Ge
Hash 250102_171626_901658_jw2AIi is impacted by 241228_231134_595452_3Dm6Ge


In [6]:
metadata[corrupted_hash]

{'data_seed': 14,
 'dataloader': {'batch_size': 1024,
  'batch_size_milestones': [1, 3, 6, 10, 15],
  'batch_size_multiplier': 2},
 'depth': 8,
 'early_stop_patience': None,
 'encoder_mode': 'best',
 'fit_batch_size': 1024,
 'lr_scheduler': {'cooldown': 1,
  'factor': 0.25,
  'min_lr': 0.0002,
  'patience': 3,
  'threshold': 0.0001,
  'type': 'ReduceLROnPlateau'},
 'max_epochs': 30,
 'model': 'AttentionClassifier',
 'model_seed': 14,
 'optimizer': {'lr': 0.01, 'type': 'Adam'},
 'train_seed': 14,
 'val_ratio': 0.33,
 'experiment_name': 'smeared_fvt_training_ensemble',
 'dataset': {'n_3b': 1000000,
  'ratio_4b': 0.5,
  'seed': 40,
  'signal_filename': 'HH4b_picoAOD.h5',
  'signal_ratio': 0.005},
 'smearing': {'hard_cutoff': False,
  'noise_scale': 1.0,
  'scale_mode': 'std',
  'seed': 40},
 'step': 2,
 'encoder_hash': '241223_045526_068830_0LZi9x',
 'aux_info_step': 2}

In [18]:
TrainingInfo.find({
    "experiment_name": "smeared_fvt_training_ensemble",
    "train_seed": 14,
    "dataset": lambda x: (x["signal_ratio"] == 0.005 and x["seed"] == 40 and x["n_3b"] == 100_0000),
})

('241228_231134_595452_3Dm6Ge', '250105_040500_856789_xSQlqd')

In [19]:
TrainingInfo.update_metadata()

2025-01-05 04:19:21,939 - INFO - Adding 0 files, removing 1 hashes
0it [00:00, ?it/s]


In [21]:
tinfo = TrainingInfo.load(corrupted_hash)
tinfo.hash

'241228_231134_595452_3Dm6Ge'

# Done!